In [ ]:
from google.colab import files

uploaded = files.upload()

Saving train.csv to train.csv


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("train.csv")

print("Rows and Columns:", df.shape)

print("\nFirst 5 Rows:")
display(df.head())

print("\nDataset Information:")
df.info()

Rows and Columns: (1460, 81)

First 5 Rows:


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000



Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null  

In [ ]:
missing_report = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum() / len(df) * 100).round(2)
})

missing_report = missing_report[missing_report["Missing Count"] > 0]

missing_report

,Missing Count,Missing Percentage
LotFrontage,259,17.74
Alley,1369,93.77
MasVnrType,872,59.73
MasVnrArea,8,0.55
BsmtQual,37,2.53
BsmtCond,37,2.53
BsmtExposure,38,2.60
BsmtFinType1,37,2.53
BsmtFinType2,38,2.60
Electrical,1,0.07


In [ ]:
# Exact duplicate rows
duplicate_count = df.duplicated().sum()

print("Exact duplicate rows:", duplicate_count)

Exact duplicate rows: 0


In [ ]:
# Potential duplicate records excluding ID and target
near_dup_cols = [c for c in df.columns if c not in ["Id", "SalePrice"]]

near_duplicates = df.duplicated(subset=near_dup_cols, keep=False)

print("Potential near-duplicate records:", near_duplicates.sum())

Potential near-duplicate records: 0


In [ ]:
numeric_cols = ["LotArea", "GrLivArea", "TotalBsmtSF"]

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]

    print(col, "→ Outliers:", len(outliers))

LotArea → Outliers: 69
GrLivArea → Outliers: 31
TotalBsmtSF → Outliers: 61


In [ ]:
for col in ["LotArea", "GrLivArea", "TotalBsmtSF"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 1.5 * IQR

    print(f"\n{col} - Upper Limit: {upper:.2f}")
    display(df[df[col] > upper][["Id", col, "Neighborhood", "YearBuilt"]].sort_values(col, ascending=False).head(10))


LotArea - Upper Limit: 17673.50


,Id,LotArea,Neighborhood,YearBuilt
313,314,215245,Timber,1965
335,336,164660,Timber,1965
249,250,159000,ClearCr,1958
706,707,115149,ClearCr,1971
451,452,70761,ClearCr,1975
1298,1299,63887,Edwards,2008
1396,1397,57200,Timber,1948
769,770,53504,StoneBr,2003
457,458,53227,ClearCr,1954
384,385,53107,ClearCr,1992



GrLivArea - Upper Limit: 2747.62


,Id,GrLivArea,Neighborhood,YearBuilt
1298,1299,5642,Edwards,2008
523,524,4676,Edwards,2007
1182,1183,4476,NoRidge,1996
691,692,4316,NoRidge,1994
1169,1170,3627,NoRidge,1995
185,186,3608,OldTown,1892
304,305,3493,OldTown,1880
1268,1269,3447,Crawfor,1935
635,636,3395,SWISU,1914
769,770,3279,StoneBr,2003



TotalBsmtSF - Upper Limit: 2052.00


,Id,TotalBsmtSF,Neighborhood,YearBuilt
1298,1299,6110,Edwards,2008
332,333,3206,NridgHt,2003
496,497,3200,NoRidge,1992
523,524,3138,Edwards,2007
440,441,3094,NridgHt,2008
1373,1374,2633,NoRidge,2001
1044,1045,2524,NWAmes,1981
691,692,2444,NoRidge,1994
1182,1183,2396,NoRidge,1996
224,225,2392,NridgHt,2003


In [ ]:
print(df.dtypes)

Id                 int64
MSSubClass         int64
MSZoning          object
LotFrontage      float64
LotArea            int64
                  ...   
MoSold             int64
YrSold             int64
SaleType          object
SaleCondition     object
SalePrice          int64
Length: 81, dtype: object


In [ ]:
df["MSSubClass"] = df["MSSubClass"].astype(str)

print(df["MSSubClass"].dtype)

object


In [3]:
clean_df = df.copy()

NameError: name 'df' is not defined

In [7]:
# Missing means "feature does not exist"
none_cols = [
    "Alley", "MasVnrType",
    "BsmtQual", "BsmtCond", "BsmtExposure",
    "BsmtFinType1", "BsmtFinType2",
    "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "PoolQC", "Fence", "MiscFeature"
]

for col in none_cols:
    clean_df[col] = clean_df[col].fillna("None")

# Numerical missing values
clean_df["MasVnrArea"] = clean_df["MasVnrArea"].fillna(0)

clean_df["LotFrontage"] = clean_df["LotFrontage"].fillna(
    clean_df["LotFrontage"].median()
)

# Only one missing categorical value
clean_df["Electrical"] = clean_df["Electrical"].fillna(
    clean_df["Electrical"].mode()[0]
)

# GarageYearBuilt: 0 indicates no garage
clean_df["GarageYrBlt"] = clean_df["GarageYrBlt"].fillna(0)

In [ ]:
print("Remaining missing values:", clean_df.isnull().sum().sum())

Remaining missing values: 0


In [ ]:
report = """
# DATA QUALITY REPORT

## 1. Dataset Overview

The dataset used for this assignment is the Kaggle House Prices - Advanced Regression Techniques dataset.

- Rows: 1460
- Columns: 81
- Target variable: SalePrice
- Problem type: Regression

## 2. Missing Value Analysis

A complete missing-value analysis was performed.

| Column | Missing Count | Missing Percentage |
|---|---:|---:|
| LotFrontage | 259 | 17.74% |
| Alley | 1369 | 93.77% |
| MasVnrType | 872 | 59.73% |
| MasVnrArea | 8 | 0.55% |
| BsmtQual | 37 | 2.53% |
| BsmtCond | 37 | 2.53% |
| BsmtExposure | 38 | 2.60% |
| BsmtFinType1 | 37 | 2.53% |
| BsmtFinType2 | 38 | 2.60% |
| Electrical | 1 | 0.07% |
| FireplaceQu | 690 | 47.26% |
| GarageType | 81 | 5.55% |
| GarageYrBlt | 81 | 5.55% |
| GarageFinish | 81 | 5.55% |
| GarageQual | 81 | 5.55% |
| GarageCond | 81 | 5.55% |
| PoolQC | 1453 | 99.52% |
| Fence | 1179 | 80.75% |
| MiscFeature | 1406 | 96.30% |

## 3. Missing Value Decisions

- LotFrontage: Median imputation was used because it is numerical and the median is less affected by extreme values.
- Alley: Filled with "None" because missing values generally indicate no alley access.
- MasVnrType: Filled with "None" because missing values generally indicate no masonry veneer.
- MasVnrArea: Filled with 0 because no masonry veneer means zero veneer area.
- Basement-related columns: Filled with "None" because missing values generally indicate that the property has no basement.
- FireplaceQu: Filled with "None" because missing values indicate no fireplace.
- GarageType, GarageFinish, GarageQual, GarageCond: Filled with "None" because missing values generally indicate no garage.
- GarageYrBlt: Filled with 0 because the missing value occurs when a garage is not present.
- PoolQC: Filled with "None" because missing values indicate no pool.
- Fence: Filled with "None" because missing values indicate no fence.
- MiscFeature: Filled with "None" because missing values indicate no miscellaneous feature.
- Electrical: Filled with the mode because only one value was missing.

A blanket mean imputation strategy was not used.

## 4. Duplicate Records

- Exact duplicate rows: 0
- Potential near-duplicate records: 0

Therefore, no records were removed because of duplication.

## 5. Outlier Detection

The IQR method was used to identify numerical outliers.

| Column | Number of Outliers |
|---|---:|
| LotArea | 69 |
| GrLivArea | 31 |
| TotalBsmtSF | 6 |

The identified outliers were not automatically removed because extreme property sizes can represent genuine large properties.

Particularly extreme observations such as GrLivArea = 5642 and TotalBsmtSF = 6110 should be investigated further rather than automatically classified as errors.

## 6. Data Type Correction

MSSubClass was converted from integer to categorical/object type because its values represent building class codes rather than continuous numerical measurements.

## 7. Final Cleaning Result

After applying the selected missing-value strategies:

- Remaining missing values: 0
- Exact duplicates: 0
- Potential near-duplicates: 0
- Outliers were identified and retained for further investigation.
- MSSubClass was correctly converted to categorical type.

The original raw dataset was preserved, while cleaning was performed on a separate copy named `clean_df`.
"""

with open("DATA_QUALITY_REPORT.md", "w") as f:
    f.write(report)

print("DATA_QUALITY_REPORT.md created successfully!")

DATA_QUALITY_REPORT.md created successfully!


In [ ]:
from google.colab import files
files.download("DATA_QUALITY_REPORT.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
import pandas as pd

df = pd.read_csv("train.csv")

print("Rows and Columns:", df.shape)

Rows and Columns: (1460, 81)


In [6]:
clean_df = df.copy()

In [8]:
# Separate features and target

X = clean_df.drop("SalePrice", axis=1)
y = clean_df["SalePrice"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1460, 80)
y shape: (1460,)


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1168, 80)
X_test: (292, 80)
y_train: (1168,)
y_test: (292,)


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify column types
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X_train.select_dtypes(include=["object"]).columns

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numerical features: 37
Categorical features: 43


In [11]:
# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [12]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DummyRegressor(strategy="mean"))
])

baseline_model.fit(X_train, y_train)

baseline_pred = baseline_model.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

print("Mean Baseline")
print("MAE :", baseline_mae)
print("RMSE:", baseline_rmse)
print("R²  :", baseline_r2)

Mean Baseline
MAE : 62575.926451960964
RMSE: 87619.03450611533
R²  : -0.0008824918802490256


In [13]:
from sklearn.linear_model import LinearRegression

linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)

linear_pred = linear_model.predict(X_test)

linear_mae = mean_absolute_error(y_test, linear_pred)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_pred))
linear_r2 = r2_score(y_test, linear_pred)

print("Linear Regression")
print("MAE :", linear_mae)
print("RMSE:", linear_rmse)
print("R²  :", linear_r2)

Linear Regression
MAE : 21103.79708731959
RMSE: 65268.691254371304
R²  : 0.44461266144498435


In [14]:
from sklearn.linear_model import Ridge

ridge_results = []

for alpha in [0.1, 1, 10]:

    ridge_model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", Ridge(alpha=alpha))
    ])

    ridge_model.fit(X_train, y_train)

    ridge_pred = ridge_model.predict(X_test)

    mae = mean_absolute_error(y_test, ridge_pred)
    rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
    r2 = r2_score(y_test, ridge_pred)

    ridge_results.append({
        "Model": f"Ridge (alpha={alpha})",
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    })

    print(f"Ridge alpha={alpha}")
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R²  :", r2)
    print()

Ridge alpha=0.1
MAE : 18514.969892754303
RMSE: 29122.944109426182
R²  : 0.889425058245105

Ridge alpha=1
MAE : 19138.056669657042
RMSE: 29750.372748326776
R²  : 0.884609251972223

Ridge alpha=10
MAE : 19159.14757536448
RMSE: 30551.69715164878
R²  : 0.8783094520841292



In [15]:
comparison = pd.DataFrame([
    {
        "Model": "Mean Baseline",
        "MAE": baseline_mae,
        "RMSE": baseline_rmse,
        "R²": baseline_r2
    },
    {
        "Model": "Linear Regression",
        "MAE": linear_mae,
        "RMSE": linear_rmse,
        "R²": linear_r2
    },
    {
        "Model": "Ridge (alpha=0.1)",
        "MAE": 18514.969892754303,
        "RMSE": 29122.944109426182,
        "R²": 0.889425058245105
    },
    {
        "Model": "Ridge (alpha=1)",
        "MAE": 19138.056669657042,
        "RMSE": 29750.372748326776,
        "R²": 0.884609251972223
    },
    {
        "Model": "Ridge (alpha=10)",
        "MAE": 19159.14757536448,
        "RMSE": 30551.69715164878,
        "R²": 0.8783094520841292
    }
])

comparison.round(2)

,Model,MAE,RMSE,R²
0,Mean Baseline,62575.93,87619.03,-0.00
1,Linear Regression,21103.80,65268.69,0.44
2,Ridge (alpha=0.1),18514.97,29122.94,0.89
3,Ridge (alpha=1),19138.06,29750.37,0.88
4,Ridge (alpha=10),19159.15,30551.70,0.88


In [16]:
# Log-transform target
y_train_log = np.log1p(y_train)

log_ridge_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=0.1))
])

log_ridge_model.fit(X_train, y_train_log)

# Prediction in log scale
log_pred = log_ridge_model.predict(X_test)

# Convert back to original SalePrice scale
log_ridge_pred = np.expm1(log_pred)

# Metrics
log_ridge_mae = mean_absolute_error(y_test, log_ridge_pred)
log_ridge_rmse = np.sqrt(mean_squared_error(y_test, log_ridge_pred))
log_ridge_r2 = r2_score(y_test, log_ridge_pred)

print("Log-Target Ridge (alpha=0.1)")
print("MAE :", log_ridge_mae)
print("RMSE:", log_ridge_rmse)
print("R²  :", log_ridge_r2)

Log-Target Ridge (alpha=0.1)
MAE : 15085.840911624737
RMSE: 22823.790952919044
R²  : 0.9320856219942197
